In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

import pathlib as pl

from tqdm.auto import tqdm

import matplotlib
matplotlib.rcParams['svg.fonttype'] = 'none'
import seaborn as sns

# Embed new model

In [ ]:
from spatialfusion.embed.embed import AEInputs, run_full_embedding

In [ ]:
sample_list = ['P1CRC','P2CRC','P5CRC','P3NAT','P5NAT']

In [ ]:
basepath = pl.Path('../../../Broad_SpatialFoundation/VisiumHD-CRC/')
ae_inputs_by_sample = {}
for sample_name in tqdm(sample_list):
    output_dir = basepath / sample_name
    virchow_df = pd.read_parquet(pl.Path(output_dir) / 'embeddings' / 'Virchow2.parquet')
    virchow_df.index = virchow_df.index.astype(str) + '::' + sample_name  
    scgpt_df = pd.read_csv(pl.Path(output_dir) / 'embeddings' / 'scGPT.csv', index_col=0)
    scgpt_df.index = scgpt_df.index.astype(str) + '::' + sample_name  

    adata = sc.read_h5ad(basepath / sample_name / 'adata.h5ad')
    adata.obs = pd.concat([adata.obs, pd.DataFrame(adata.obsm['spatial'], index=adata.obs_names, columns=['X_coord','Y_coord'])],axis=1)
    adata.obs["sample_id"] = sample_name
    adata.obs_names = adata.obs_names.astype(str) + '::' + sample_name  

    ae_inputs_by_sample[sample_name] = AEInputs(adata=adata, z_uni=virchow_df, z_scgpt=scgpt_df)


In [ ]:
# this uses the average version
all_embeddings = run_full_embedding(
    ae_inputs_by_sample=ae_inputs_by_sample,
    ae_model_path='../../../SpatialFusion/results/ae_encoder_sweep/virchow_scgpt_full_20260602-065430_e436487d/model.pt',
    gcn_model_path='../../../SpatialFusion/results/gcn_fusion_sweep/virchow_scgpt_full_gcn_avg_reg_cls_6c8c309d/model.pt',
    device="cuda:6",
    combine_mode="average",
    spatial_key='spatial',
    k=30,
    celltype_key="cellsubtypes",
    save_ae_dir=None,  # optional
)

In [ ]:
all_embeddings.to_parquet('virchow_scgpt_SpatialFusion_avg_full_cls.parquet')

# Now get clusters

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
import pathlib as pl
import scanpy as sc
import pandas as pd
import numpy as np
from tqdm import tqdm

def group_small_clusters(
    df: pd.DataFrame,
    cluster_col: str,
    min_count: int = 1000,
    new_label: str = "small_clusters",
    output_col: str = None
) -> pd.DataFrame:
    """
    Groups small clusters in a DataFrame column into a single label.

    Parameters:
        df (pd.DataFrame): Input DataFrame containing cluster labels.
        cluster_col (str): Name of the column containing cluster labels (e.g., 'leiden').
        min_count (int): Minimum number of entries a cluster must have to avoid grouping.
        new_label (str): Label to assign to small clusters.
        output_col (str or None): Name of the new column to store grouped labels. 
                                  If None, defaults to '{cluster_col}_grouped'.

    Returns:
        pd.DataFrame: A copy of the DataFrame with a new column containing grouped cluster labels.
    """
    if cluster_col not in df.columns:
        raise ValueError(f"Column '{cluster_col}' not found in DataFrame.")

    output_col = output_col or f"{cluster_col}_grouped"
    cluster_counts = df[cluster_col].value_counts()
    small_clusters = cluster_counts[cluster_counts < min_count].index

    new_df = df.copy()
    new_df[output_col] = df[cluster_col].astype(str)
    new_df.loc[df[cluster_col].isin(small_clusters), output_col] = new_label

    return new_df[output_col]


In [ ]:
sample_list = ['P1CRC','P2CRC','P5CRC','P3NAT','P5NAT']
base_dir = pl.Path('../../../Broad_SpatialFoundation/VisiumHD-CRC/')

In [ ]:
all_embeddings_virchow = pd.read_parquet('virchow_scgpt_SpatialFusion_avg_full_cls.parquet').set_index('cell_id')
all_embeddings_uni = pd.read_parquet('../../../Broad_SpatialFoundation/VisiumHD-CRC/SpatialFusion_full_emb.parquet').set_index('cell_id')

In [ ]:
adatas = []

for sample in tqdm(sample_list):
    adata = sc.read_h5ad(base_dir / sample / "adata.h5ad")

    if "X_cnv" in adata.obsm:
        del adata.obsm["X_cnv"]

    adata.obs_names = adata.obs_names + "::" + sample
    adata.obs["sample_id"] = sample

    # UNI embeddings
    mask = adata.obs_names.isin(all_embeddings_uni.index)
    adata = adata[mask, :].copy()

    embeddings_df = all_embeddings_uni.loc[adata.obs_names]

    adata.obsm["SpatialFusion_UNI"] = (
        embeddings_df[[str(i) for i in range(10)]].to_numpy()
    )

    # Virchow embeddings
    mask = adata.obs_names.isin(all_embeddings_virchow.index)
    adata = adata[mask, :].copy()

    embeddings_virchow_df = all_embeddings_virchow.loc[adata.obs_names]

    adata.obsm["SpatialFusion_Virchow"] = (
        embeddings_virchow_df[[str(i) for i in range(10)]].to_numpy()
    )

    print(
        adata.shape,
        adata.obsm["SpatialFusion_UNI"].shape,
        adata.obsm["SpatialFusion_Virchow"].shape,
    )

    adatas.append(adata)

In [ ]:
adata = adatas[0].concatenate(*adatas[1:])
adata.obs_names = adata.obs_names.str.split('-').str[0]

In [ ]:
adata = adata[adata.obs.celltypes != 'Noise'].copy()

In [ ]:
adata_obs = pd.read_csv('../../../Broad_SpatialFoundation/notebooks/full_CRC_obs.csv', index_col=0)

In [ ]:
adata = adata[adata_obs.index].copy()

In [ ]:
adata.obs['UNI_leiden'] = adata_obs['leiden']

In [ ]:
adata.obs['UNI_leiden_joint'] = group_small_clusters(
    adata.obs[['UNI_leiden']],
    cluster_col='UNI_leiden',
    min_count= 500,
    new_label= "Other",
    output_col = None
)

In [ ]:
print(
    "Number of clusters:",
    adata.obs["UNI_leiden_joint"].nunique()
)

In [ ]:
adata.obs = pd.concat([adata.obs, adata_obs[['refined_celltypes','refined_cellsubtypes']]],axis=1)

# How well do we recover clusters 1 and 2 with Virchow

In [ ]:
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import normalized_mutual_info_score

In [ ]:
sc.pp.neighbors(adata, use_rep='SpatialFusion_Virchow')

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)

# ------------------------------------
# Recovery score
# ------------------------------------

def recovery_score(
    uni_labels,
    vir_labels,
    uni_cluster,
    k=2
):
    """
    Fraction of cells in a UNI cluster
    recovered by the top-k overlapping
    Virchow clusters.
    """

    mask = uni_labels.astype(str) == str(uni_cluster)

    vir_counts = (
        vir_labels[mask]
        .value_counts(normalize=True)
        .sort_values(ascending=False)
    )

    return vir_counts.iloc[:k].sum()


# ------------------------------------
# Resolution sweep
# ------------------------------------

resolutions = np.arange(0.1, 0.35, 0.05)

results = []

for res in tqdm(resolutions):

    print(f"Running resolution {res:.2f}")

    sc.tl.leiden(
        adata,
        resolution=res,
        key_added="Vir_tmp",
        flavor="igraph",
        n_iterations=2
    )

    adata.obs["Vir_tmp_joint"] = group_small_clusters(
        adata.obs[["Vir_tmp"]],
        cluster_col="Vir_tmp",
        min_count=500,
        new_label="Other",
        output_col=None
    )

    # ------------------------------
    # Recovery of UNI clusters
    # ------------------------------

    rec1 = recovery_score(
        adata.obs["UNI_leiden_joint"],
        adata.obs["Vir_tmp_joint"],
        uni_cluster="1",
        k=2
    )

    rec2 = recovery_score(
        adata.obs["UNI_leiden_joint"],
        adata.obs["Vir_tmp_joint"],
        uni_cluster="2",
        k=2
    )

    # ------------------------------
    # Global agreement
    # ------------------------------

    ari = adjusted_rand_score(
        adata.obs["UNI_leiden_joint"],
        adata.obs["Vir_tmp_joint"]
    )

    nmi = normalized_mutual_info_score(
        adata.obs["UNI_leiden_joint"],
        adata.obs["Vir_tmp_joint"]
    )

    results.append({
        "resolution": res,
        "n_clusters": adata.obs["Vir_tmp_joint"].nunique(),
        "cluster1_recovery": rec1,
        "cluster2_recovery": rec2,
        "mean_recovery": np.mean([rec1, rec2]),
        "ARI": ari,
        "NMI": nmi
    })

results = pd.DataFrame(results)

results.sort_values(
    "mean_recovery",
    ascending=False
).head()

In [ ]:
plt.figure(figsize=(6,4))

plt.plot(
    results["resolution"],
    results["cluster1_recovery"],
    label="UNI cluster 1"
)

plt.plot(
    results["resolution"],
    results["cluster2_recovery"],
    label="UNI cluster 2"
)

plt.plot(
    results["resolution"],
    results["mean_recovery"],
    lw=3,
    label="Mean"
)

plt.ylabel("Best overlap")
plt.xlabel("Virchow Leiden resolution")
plt.legend()

# Compare clusterings

## Res 0.1

In [ ]:
sc.tl.leiden(
    adata,
    resolution=0.1,
    key_added="Virchow_leiden",
    flavor="igraph",
    n_iterations=2
)

In [ ]:
adata.obs["Virchow_leiden"] = (
    adata.obs["Virchow_leiden"].astype("category")
)

print(
    "Number of clusters:",
    adata.obs["Virchow_leiden"].nunique()
)

In [ ]:
adata.obs['Virchow_leiden_joint'] = group_small_clusters(
    adata.obs[['Virchow_leiden']],
    cluster_col='Virchow_leiden',
    min_count= 500,
    new_label= "Other",
    output_col = None
)

In [ ]:
print(
    "Number of clusters:",
    adata.obs["Virchow_leiden_joint"].nunique()
)

In [ ]:
cm = pd.crosstab(
    adata.obs['UNI_leiden_joint'],
    adata.obs['Virchow_leiden_joint']
)

cm_uni = cm.div(cm.sum(axis=0), axis=1)

plt.figure(figsize=(9,8))
sns.heatmap(cm_uni, annot=True, fmt='.2f')
plt.ylabel('UNI cluster')
plt.xlabel('Virchow cluster')
plt.title('Proportion of Virchow cluster assigned to each UNI cluster')

In [ ]:
adata.obs["UNI_leiden_joint"] = (
    adata.obs["UNI_leiden_joint"].astype(str).astype("category")
)

adata.obs["Virchow_leiden_joint"] = (
    adata.obs["Virchow_leiden_joint"].astype(str).astype("category")
)

In [ ]:
import seaborn as sns

all_clusters = sorted(
    set(adata.obs["UNI_leiden_joint"].cat.categories)
    |
    set(adata.obs["Virchow_leiden_joint"].cat.categories)
)

palette = sns.color_palette(
    "tab20",
    n_colors=len(all_clusters)
)

cluster_colors = {
    cl: palette[i]
    for i, cl in enumerate(all_clusters)
}

In [ ]:
import matplotlib.pyplot as plt

samples = [
    "P1CRC",
    "P2CRC",
    "P5CRC",
    "P3NAT",
    "P5NAT"
]

fig, axes = plt.subplots(
    1,
    len(samples),
    figsize=(20,4)
)

for ax, sample in zip(axes, samples):

    ad = adata[adata.obs.sample_id == sample]

    sc.pl.embedding(
        ad,
        basis="spatial",
        color="UNI_leiden_joint",
        palette=cluster_colors,
        ax=ax,
        show=False,
        frameon=False,
        size=8,
        legend_loc=None
    )

    ax.set_title(sample)

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(
    1,
    len(samples),
    figsize=(20,4)
)

for ax, sample in zip(axes, samples):

    ad = adata[adata.obs.sample_id == sample]

    sc.pl.embedding(
        ad,
        basis="spatial",
        color="Virchow_leiden_joint",
        palette=cluster_colors,
        ax=ax,
        show=False,
        frameon=False,
        size=8,
        legend_loc=None
    )

    ax.set_title(sample)

plt.tight_layout()

## Res 0.15  MAIN RES

In [ ]:
sc.tl.leiden(
    adata,
    resolution=0.15,
    key_added="Virchow_leiden",
    flavor="igraph",
    n_iterations=2
)

In [ ]:
adata.obs["Virchow_leiden"] = (
    adata.obs["Virchow_leiden"].astype("category")
)

print(
    "Number of clusters:",
    adata.obs["Virchow_leiden"].nunique()
)

In [ ]:
adata.obs['Virchow_leiden_joint'] = group_small_clusters(
    adata.obs[['Virchow_leiden']],
    cluster_col='Virchow_leiden',
    min_count= 500,
    new_label= "Other",
    output_col = None
)

In [ ]:
print(
    "Number of clusters:",
    adata.obs["Virchow_leiden_joint"].nunique()
)

In [ ]:
cm = pd.crosstab(
    adata.obs['UNI_leiden_joint'],
    adata.obs['Virchow_leiden_joint']
)

cm_uni = cm.div(cm.sum(axis=0), axis=1)

plt.figure(figsize=(9,8))
sns.heatmap(cm_uni, annot=True, fmt='.2f')
plt.ylabel('UNI cluster')
plt.xlabel('Virchow cluster')
plt.title('Proportion of Virchow cluster assigned to each UNI cluster')

In [ ]:
adata.obs["UNI_leiden_joint"] = (
    adata.obs["UNI_leiden_joint"].astype(str).astype("category")
)

adata.obs["Virchow_leiden_joint"] = (
    adata.obs["Virchow_leiden_joint"].astype(str).astype("category")
)

In [ ]:
import seaborn as sns

all_clusters = sorted(
    set(adata.obs["UNI_leiden_joint"].cat.categories)
    |
    set(adata.obs["Virchow_leiden_joint"].cat.categories)
)

palette = sns.color_palette(
    "tab20",
    n_colors=len(all_clusters)
)

cluster_colors = {
    cl: palette[i]
    for i, cl in enumerate(all_clusters)
}

In [ ]:
fig, axes = plt.subplots(
    1,
    len(samples),
    figsize=(20,4)
)

for ax, sample in zip(axes, samples):

    ad = adata[adata.obs.sample_id == sample]

    sc.pl.embedding(
        ad,
        basis="spatial",
        color="Virchow_leiden_joint",
        palette=cluster_colors,
        ax=ax,
        show=False,
        frameon=False,
        size=8,
        legend_loc=None
    )

    ax.set_title(sample)

plt.tight_layout()

## Do we find the same GEX difference

In [ ]:
# 1 UNI --> 15/16
# 2 UNI --> 6

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# --- Data prep (your same logic) ---
cl_order = np.sort(np.setdiff1d(adata.obs.Virchow_leiden_joint.unique(), ['Other', 'nan']).astype(int)).astype(str)
sample_order = ['P1CRC','P2CRC','P5CRC','P3NAT','P5NAT']

vc = adata.obs[['sample_id','Virchow_leiden_joint']].value_counts().unstack().T
vc = vc / vc.sum(axis=0)
vc = vc.loc[cl_order, sample_order]

# --- Style setup (Nature Genetics–like aesthetic) ---
sns.set_theme(context="talk", style="white")

# --- Custom colormap: subtle, elegant red-to-gray gradient ---
cmap = LinearSegmentedColormap.from_list(
    "vlag_redgray",
    ["#f7f7f7", "#f4a3a8", "#b40426"]
)

# --- Prepare annotation matrix ---
annot = vc.copy() * 100  # convert to percent
annot_fmt = annot.copy()

# format as strings with rules:
for i in range(annot_fmt.shape[0]):
    for j in range(annot_fmt.shape[1]):
        val = annot_fmt.iat[i, j]
        if pd.isna(val):
            annot_fmt.iat[i, j] = "N.A."
        elif val < 1:
            annot_fmt.iat[i, j] = "<1%"
        else:
            annot_fmt.iat[i, j] = f"{val:.0f}%"  # round to nearest percent

# --- Plot ---
fig, ax = plt.subplots(figsize=(2, 3.5), dpi=300)

sns.heatmap(
    vc,
    cmap=cmap,
    annot=annot_fmt,
    fmt="",
    linewidths=0.4,
    linecolor="white",
    cbar=False,
    annot_kws={"fontsize": 7, "color": "black"},
    ax=ax,
)

# --- Aesthetic adjustments ---
ax.set_xlabel("", fontsize=11)
ax.set_ylabel("", fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=7)
ax.set_yticks(np.arange(len(vc.index)) + 0.5)
ax.set_yticklabels(vc.index, rotation=0, fontsize=7)
ax.tick_params(length=0)

# Remove borders and extra gridlines
for spine in ax.spines.values():
    spine.set_visible(False)

# Optional title
ax.set_title("", fontsize=12, pad=10, fontweight="normal")

plt.tight_layout()
#fig.savefig('../../../SpatialFusion/results/figures_Fig5/niches_proportions.svg')
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, Sequence, Mapping, Tuple, Dict, List
from matplotlib.colors import to_hex, to_rgb
import colorsys


def plot_cluster_composition_stacked(
    df,
    cluster_key: str = "leiden",
    type_key: str = "celltypes",                 # or "cellsubtypes"
    cluster_order: Optional[Sequence[str]] = None,
    strict_order: bool = False,                  # if True, only show clusters in cluster_order
    palette: Optional[Mapping[str, str]] = None, # dict {celltype: color}; auto if None
    top_types: Optional[int] = None,             # keep top N types globally, rest→"Other"
    min_frac: Optional[float] = None,            # keep types with global frac >= min_frac, rest→"Other"
    other_label: str = "Other",
    type_order: Optional[Sequence[str]] = None,  # custom order of stack segments
    figsize=(10, 5),
    percent_labels: bool = False,                # print % on bars
    label_threshold: float = 0.05,               # only label segments >=5%
    savefig: Optional[str] = None,
):
    """
    Plot a 100% stacked barplot of type proportions per cluster using adata.obs.
    Returns a long-form DataFrame with columns: [cluster, type, count, frac, percent]
    """
    obs = df[[cluster_key, type_key]].dropna().copy()
    obs[cluster_key] = obs[cluster_key].astype(str)
    obs[type_key]    = obs[type_key].astype(str)

    # Cross-tab counts (rows=clusters, cols=types)
    ct = pd.crosstab(obs[cluster_key], obs[type_key])

    # Global filtering of rare types (optional)
    keep_cols = ct.columns.tolist()
    if top_types is not None:
        keep_cols = (
            ct.sum(axis=0)
              .sort_values(ascending=False)
              .head(top_types)
              .index.tolist()
        )
    if min_frac is not None:
        global_frac = ct.sum(axis=0) / ct.values.sum()
        keep_cols = sorted(set(keep_cols) | set(global_frac[global_frac >= min_frac].index.tolist()))
    if (top_types is not None) or (min_frac is not None):
        other = ct.drop(columns=keep_cols, errors="ignore").sum(axis=1)
        ct = ct[keep_cols].copy()
        if (other > 0).any():
            ct[other_label] = other
        # make sure "Other" is last
        ct = ct[[c for c in ct.columns if c != other_label] + ([other_label] if other_label in ct.columns else [])]

    # Normalize rows to 1.0 (100%)
    row_sums = ct.sum(axis=1).replace(0, np.nan)
    props = ct.div(row_sums, axis=0).fillna(0.0)

    # Cluster reordering
    if cluster_order is not None:
        cluster_order = [str(c) for c in cluster_order]
        missing = [c for c in cluster_order if c not in props.index]
        if strict_order:
            props = props.reindex(cluster_order).dropna(how="all")
        else:
            extras = [c for c in props.index if c not in cluster_order]
            props = props.reindex(cluster_order + extras)
        if missing:
            print(f"Warning: these clusters from cluster_order were not found and will be skipped: {missing}")
    else:
        props = props.sort_index()

    if props.empty:
        raise ValueError("No clusters to plot after filtering/reordering.")

    # Determine stack (type) order
    types_order = props.columns.tolist()
    if type_order is not None:
        type_order = [t for t in type_order if t in props.columns]
        leftovers = [t for t in props.columns if t not in type_order]
        types_order = type_order + leftovers

    # Build color map
    if palette is None:
        base = sns.color_palette("tab10", n_colors=max(10, len(types_order)))
        colmap = dict(zip(types_order, base[:len(types_order)]))
        if other_label in types_order:
            colmap[other_label] = "#B0B0B0"  # gray for "Other"
    else:
        colmap = {t: palette.get(t, "#BBBBBB") for t in types_order}

    # Plot (stacked bars)
    plt.figure(figsize=figsize)
    bottom = np.zeros(len(props))
    x = np.arange(len(props.index))
    ax = plt.gca()

    for t in types_order:
        vals = props[t].values
        ax.bar(x, vals, bottom=bottom, width=0.9, color=colmap[t], label=t, edgecolor="none")
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(props.index, rotation=45, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Composition (% of cells)")
    ax.set_xlabel(cluster_key)
    ax.set_title(f"{type_key} composition per {cluster_key}")

    if percent_labels:
        for i, cl in enumerate(props.index):
            cum = 0.0
            for t in types_order:
                h = props.loc[cl, t]
                if h >= label_threshold:
                    ax.text(i, cum + h/2, f"{h*100:.0f}%", ha="center", va="center", fontsize=8, color="white")
                cum += h

    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title=type_key)
    plt.tight_layout()

    if savefig:
        plt.savefig(savefig, dpi=200, bbox_inches='tight')
    plt.show()

    # Long-form result for downstream use
    plot_df = (
        props.reset_index()
             .melt(id_vars=cluster_key, var_name="type", value_name="frac")
             .rename(columns={cluster_key: "cluster"})
    )
    plot_df["percent"] = (plot_df["frac"] * 100).round(2)
    counts_long = (
        ct.reset_index()
          .melt(id_vars=cluster_key, var_name="type", value_name="count")
          .rename(columns={cluster_key: "cluster"})
    )
    plot_df = plot_df.merge(counts_long, on=["cluster", "type"], how="left")
    return plot_df

def _lightness_shades(base_color, n, l_low=0.35, l_high=0.85):
    """Generate a sequence of lighter shades from a base color."""
    rgb = to_rgb(base_color)
    h, l, s = colorsys.rgb_to_hls(*rgb)
    ls = np.linspace(l_low, l_high, n)
    shades = [to_hex(colorsys.hls_to_rgb(h, li, s)) for li in ls]
    return shades

def make_hierarchical_palettes(
    df,
    parent_key: str = "celltypes",
    child_key: str  = "cellsubtypes",
    parent_order: Optional[Sequence[str]] = None,
    child_order: str = "alpha",
    base_palette: Optional[Mapping[str, str]] = None,  # legacy param
    parent_palette_dict: Optional[Dict[str, Tuple[float, float, float]]] = None,  # ✅ new
    unknown_parent_color: str = "#9e9e9e",
    shade_lightness: Tuple[float, float] = (0.35, 0.85),
) -> Tuple[Dict[str, str], Dict[str, str], List[str], List[str]]:
    """
    Generate hierarchical palettes with optional parent palette override.

    Parameters
    ----------
    parent_palette_dict : dict, optional
        If provided, should be {parent_label: color (hex or RGB)}.
        These colors are used directly for parent_palette and for shading.

    Returns
    -------
    parent_palette : {parent -> hex}
    child_palette  : {child -> hex}
    parent_order_out : list of parents
    child_order_out  : list of children grouped by parent
    """
    obs = df[[parent_key, child_key]].copy()
    obs[parent_key] = obs[parent_key].astype(str)
    obs[child_key]  = obs[child_key].astype(str)

    # --- Parent order ---
    parents = obs[parent_key].unique().tolist()
    if parent_order is not None:
        parent_order_out = [p for p in parent_order if p in parents] + \
                           [p for p in parents if p not in parent_order]
    else:
        freq = obs[parent_key].value_counts()
        parent_order_out = freq.index.tolist() + [p for p in parents if p not in freq.index]

    # --- Children per parent ---
    children_per_parent = {}
    for p in parents:
        sub = obs.loc[obs[parent_key] == p, child_key]
        if child_order == "freq":
            children = sub.value_counts().index.tolist()
        else:
            children = sorted(sub.unique().tolist())
        children_per_parent[p] = children

    # --- Build parent colors ---
    if parent_palette_dict is not None:
        # ✅ use directly, fallback to gray if missing
        parent_palette = {
            p: to_hex(parent_palette_dict.get(p, unknown_parent_color))
            for p in parent_order_out
        }
    elif base_palette is not None:
        parent_palette = {p: base_palette.get(p, unknown_parent_color) for p in parent_order_out}
    else:
        n_par = len(parent_order_out)
        base = sns.color_palette("tab10" if n_par <= 10 else "hls", n_colors=n_par)
        parent_palette = {p: to_hex(base[i]) for i, p in enumerate(parent_order_out)}

    # --- Build child colors as shades of parent ---
    l_low, l_high = shade_lightness
    child_palette = {}
    child_order_out = []
    for p in parent_order_out:
        base_col = parent_palette.get(p, unknown_parent_color)
        kids = children_per_parent.get(p, [])
        if not kids:
            continue
        shades = _lightness_shades(base_col, len(kids), l_low, l_high)
        for k, col in zip(kids, shades):
            child_palette[k] = col
        child_order_out.extend(kids)

    return parent_palette, child_palette, parent_order_out, child_order_out



In [ ]:
def build_palettes_from_adata(adata, palette_specs):
    """
    Build labeled color palettes for categorical columns in adata.obs.

    Parameters
    ----------
    adata : AnnData
        Must have .obs DataFrame containing categorical columns.
    palette_specs : dict
        Mapping {column_name: palette} where palette can be:
          - a string palette name (e.g. "tab10")
          - a list of RGB colors (custom)

    Returns
    -------
    dict
        {column_name: {label: color}} mapping.
    """
    custom_palettes = {}

    for col, palette in palette_specs.items():
        if col not in adata.obs.columns:
            print(f"⚠️ Warning: '{col}' not found in adata.obs — skipping.")
            continue

        unique_vals = sorted(adata.obs[col].astype(str).dropna().unique())
        n_unique = len(unique_vals)

        # If user passed a name → generate via seaborn
        if isinstance(palette, str):
            pal_colors = sns.color_palette(palette, n_colors=n_unique)
        # If user passed a list → use directly
        elif isinstance(palette, (list, tuple)):
            pal_colors = palette[:n_unique]
        else:
            raise ValueError(f"Unsupported palette type for '{col}': {type(palette)}")

        color_dict = dict(zip(unique_vals, pal_colors))
        custom_palettes[col] = color_dict

    print(f"✅ Built palettes for {len(custom_palettes)} columns.")
    return custom_palettes


def plot_celltype_spatial_single_split_legend(
    df,
    color_by="celltype",
    sample_id=None,
    title=None,
    palette_dict=None,         # ✅ added
    palette_name="tab20",
    s=1.5,
    save_svg=True,
    output_prefix="spatial_plot",
    legend_title=None,
):
    """
    Nature Genetics–style spatial scatterplot for one sample,
    saving main plot as PNG (raster) and legend separately as SVG (vector).
    """
    sns.set_style("white")
    sns.set_context("talk")

    # --- Subset one sample ---
    if sample_id is not None:
        df = df[df["sample_id"] == sample_id].copy()
        if df.empty:
            raise ValueError(f"Sample ID '{sample_id}' not found in DataFrame.")

    # --- Colors ---
    unique_labels = sorted(df[color_by].dropna().unique())
    if palette_dict is not None and color_by in palette_dict:
        print('Using provided color palette.')
        color_dict = palette_dict[color_by]
    else:
        print('Generating color palette.')
        palette = sns.color_palette(palette_name, n_colors=len(unique_labels))
        color_dict = dict(zip(unique_labels, palette))

    # --- Main plot ---
    fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
    sns.scatterplot(
        data=df,
        x="X_coord", y="Y_coord",
        hue=color_by, palette=color_dict,
        s=s, alpha=0.9, linewidth=0,
        rasterized=True, ax=ax, legend=False
    )
    ax.invert_yaxis(); ax.set_aspect("equal", adjustable="box")
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(""); ax.set_ylabel("")
    plt.tight_layout()

    # --- Save main figure ---
    fname_main = f"{output_prefix}_{sample_id or 'sample'}_main.png"
    fig.savefig(fname_main, dpi=300, bbox_inches="tight", transparent=True, format="png")
    print(f"Saved main figure: {fname_main}")

    # --- Legend ---
    fig_leg, ax_leg = plt.subplots(figsize=(3, 0.5 * len(unique_labels)), dpi=300)
    handles = [
        plt.Line2D([0], [0], marker='o', color='none', label=label,
                   markerfacecolor=color_dict[label], markersize=8)
        for label in unique_labels
    ]
    ax_leg.legend(handles=handles, loc="center left", frameon=False,
                  title=legend_title or color_by, title_fontsize=14, fontsize=14)
    ax_leg.axis("off")
    plt.tight_layout()

    if save_svg:
        fname_leg = f"{output_prefix}_{sample_id or 'sample'}_legend.svg"
        fig_leg.savefig(fname_leg, dpi=300, bbox_inches="tight", transparent=True, format="svg")
        print(f"Saved legend: {fname_leg}")

    plt.close(fig); plt.close(fig_leg)


In [ ]:
tab_filtered = sns.color_palette()
tab_filtered = [c for i,c in enumerate(tab_filtered) if i not in [4,6]]

tab20_filtered = sns.color_palette('tab20') + sns.color_palette('tab20c')[:17]
tab20_filtered = [c for i,c in enumerate(tab20_filtered) if i not in [8,9,12,13]]

In [ ]:
palette_specs = {
            'Virchow_leiden_joint': tab20_filtered,
            'refined_cellsubtypes': tab20_filtered,
            'refined_celltypes': tab_filtered,
        }

palette_dict_1 = build_palettes_from_adata(adata, palette_specs)

In [ ]:
# Build coherent palettes & orders
parent_pal, child_pal, parent_order, child_order = make_hierarchical_palettes(
    adata.obs,
    parent_key="refined_celltypes",
    child_key="refined_cellsubtypes",
    parent_palette_dict=palette_dict_1['refined_celltypes'],
    child_order="alpha",                 # or "freq"
    shade_lightness=(0.35, 0.85)
)

# (A) 100% stacked bars by lineage (coarse)
plot_df_types = plot_cluster_composition_stacked(
    adata.obs,
    cluster_key="Virchow_leiden_joint",
    type_key="refined_celltypes",
    #cluster_order=[f"{i}" for i in range(18)],  # your preferred cluster order
    strict_order=False,
    palette=parent_pal,
    percent_labels=True,
    figsize=(10, 4),
    savefig='Virchow_CRC_major_celltype_stacked_finetuned_barplot.svg',
)

# (B) 100% stacked bars by subtypes (fine), colors are shades within lineage color
plot_df_subtypes = plot_cluster_composition_stacked(
    adata.obs,
    cluster_key="Virchow_leiden_joint",
    type_key="refined_cellsubtypes",
    #cluster_order=[f"{i}" for i in range(18)],
    strict_order=False,
    palette=child_pal,
    # NEW: order subtypes grouped by their parent lineage
    type_order=child_order,              # requires tiny tweak shown above
    percent_labels=True,
    figsize=(10, 6),
    savefig='Virchow_CRC_minor_celltype_stacked_finetuned_barplot.svg',
)


## Difference in GEX

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import os

def plot_umap_nature_style(
    adata,
    color_vars=["leiden", "condition", "refined_cellsubtypes", "sample_id"],
    palettes=None,
    ncols=2,
    point_size=8,
    cmap="viridis",
    frameon=False,
    figsize=(10, 10),
    title_fontsize=11,
    label_fontsize=9,
    shuffle=True,
    random_state=42,
    save_path=None,         # <--- single output SVG/PDF path
    rasterized=True,        # <--- rasterize only scatter points
):
    """
    Nature Genetics–style multi-panel UMAP figure.
    Combines multiple UMAPs into a single grid layout and rasterizes points
    while keeping vector graphics for axes/labels.

    Parameters
    ----------
    adata : AnnData
        The annotated data matrix.
    color_vars : list of str
        Variables to color by (each will be a subplot).
    palettes : dict
        Optional dict mapping color_var -> palette dict or list.
    ncols : int
        Number of columns in the subplot grid.
    save_path : str
        Output path to save the combined figure (e.g., 'umap_panels.svg').
    rasterized : bool
        If True, only the scatter points are rasterized to reduce file size.
    """

    # -------------------------------
    # Global style configuration
    # -------------------------------
    sc.set_figure_params(
        dpi=200,
        dpi_save=300,
        fontsize=9,
        facecolor="white",
        figsize=figsize,
    )

    plt.rcParams.update({
        "axes.edgecolor": "black",
        "axes.linewidth": 0.6,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "axes.titlesize": title_fontsize,
        "axes.labelsize": label_fontsize,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
    })

    # Shuffle order of cells to avoid overlay bias
    if shuffle:
        rng = np.random.default_rng(random_state)
        idx = rng.permutation(adata.n_obs)
    else:
        idx = np.arange(adata.n_obs)
    adata_shuffled = adata[idx, :].copy()

    # -------------------------------
    # Layout setup
    # -------------------------------
    n_panels = len(color_vars)
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    axes = axes.flatten()

    # -------------------------------
    # Plot each UMAP panel
    # -------------------------------
    for i, c in enumerate(color_vars):
        ax = axes[i]
        pal = palettes[c] if palettes and c in palettes else None

        # Use Scanpy scatter with matplotlib ax to control rasterization
        sc.pl.umap(
            adata_shuffled,
            color=c,
            size=point_size,
            palette=pal,
            cmap=cmap if pal is None else None,
            frameon=frameon,
            ax=ax,
            show=False,
            title=c,
        )

        # Rasterize only the scatter artists (points)
        if rasterized:
            for coll in ax.collections:
                coll.set_rasterized(True)

        ax.set_xlabel("UMAP1")
        ax.set_ylabel("UMAP2")

    # Remove empty subplots if needed
    for j in range(n_panels, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout(w_pad=0.5, h_pad=0.7)

    # -------------------------------
    # Save figure
    # -------------------------------
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=300, bbox_inches="tight", transparent=True)
        print(f"✅ Saved combined figure: {save_path}")

    plt.show()
    return fig


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.path import Path
from matplotlib.patches import PathPatch, Rectangle
from typing import Optional, Tuple, Dict, Sequence
import os

def _ribbon_path(x0, x1, y0_bot, y0_top, y1_bot, y1_top, curvature=0.5, steps=40):
    xs = np.linspace(x0, x1, steps)
    t = np.linspace(0, 1, steps)
    s = 3*t**2 - 2*t**3
    s = (1 - curvature) * t + curvature * s
    top = (1 - s) * y0_top + s * y1_top
    bot = (1 - s) * y0_bot + s * y1_bot
    verts = np.concatenate([np.column_stack([xs, bot]), np.column_stack([xs[::-1], top[::-1]])])
    codes = np.array([Path.MOVETO] + [Path.LINETO]*(len(xs)-1) + [Path.LINETO]*len(xs) + [Path.CLOSEPOLY])
    verts = np.vstack([verts, verts[0]])
    return Path(verts, codes)

def alluvial_multi_groups(
    data: pd.DataFrame,
    group_col: str = "niche",
    subtype_col: str = "cell_subtype",
    weight_col: Optional[str] = None,
    normalize: bool = True,
    min_frac_to_label: float = 0.06,
    title: Optional[str] = None,
    outfile_prefix: Optional[str] = None,
    palette: Optional[Dict[str, str]] = None,
    subtype_order: Optional[Sequence[str]] = None,
    group_order: Optional[Sequence[str]] = None,
    figsize=(8, 4),
    curvature: float = 0.6,
) -> Tuple[plt.Figure, plt.Axes, pd.DataFrame, pd.DataFrame]:
    """Generalized alluvial plot for an arbitrary number of groups."""
    groups = list(data[group_col].astype(str).unique())
    if group_order is not None:
        missing = [g for g in group_order if g not in groups]
        if missing:
            raise ValueError(f"These groups not found in data: {missing}")
        groups = list(group_order)
    else:
        groups = sorted(groups)

    if len(groups) < 2:
        raise ValueError("At least two groups are required for an alluvial plot.")

    if weight_col is None:
        data = data.copy()
        data["_w"] = 1.0
        weight_col = "_w"

    # Aggregate counts
    counts = (
        data.groupby([group_col, subtype_col])[weight_col].sum()
        .unstack(fill_value=0.0)
        .loc[groups]
    )

    # Subtype order
    if subtype_order is not None:
        present = [s for s in subtype_order if s in counts.columns]
        missing = [s for s in counts.columns if s not in present]
        ordered_cols = present + missing
    else:
        ordered_cols = counts.sum(axis=0).sort_values(ascending=False).index.tolist()

    counts = counts[ordered_cols]
    totals = counts.sum(axis=1)
    props = counts.div(totals.values[:, None]).fillna(0.0)

    # Plot setup
    fig, ax = plt.subplots(figsize=figsize, dpi=150)
    n_groups = len(groups)
    x_positions = np.linspace(0, 1, n_groups)
    bar_width = 0.1
    gap = 0.02

    # Colors
    subtype_colors = {}
    for s in counts.columns:
        if palette and s in palette:
            subtype_colors[s] = palette[s]
        else:
            subtype_colors[s] = ax._get_lines.get_next_color()

    # Compute y boundaries per group
    y_positions = {}
    for gi, g in enumerate(groups):
        y_positions[g] = {}
        y0 = 0.0
        for s in counts.columns:
            h = props.loc[g, s] if normalize else counts.loc[g, s] / totals.loc[g]
            y_positions[g][s] = (y0, y0 + h)
            y0 += h

    # Draw bars
    for gi, g in enumerate(groups):
        x = x_positions[gi]
        ax.add_patch(Rectangle((x - bar_width/2, 0), bar_width, 1.0,
                               fill=False, lw=0.5))
        for s in counts.columns:
            yb, yt = y_positions[g][s]
            ax.add_patch(Rectangle((x - bar_width/2, yb),
                                   bar_width, yt - yb,
                                   facecolor=subtype_colors[s],
                                   edgecolor='none'))

    # Draw ribbons between consecutive groups
    for gi in range(n_groups - 1):
        g0, g1 = groups[gi], groups[gi + 1]
        x0, x1 = x_positions[gi] + bar_width/2 + gap, x_positions[gi+1] - bar_width/2 - gap

        for s in counts.columns:
            c = subtype_colors[s]
            y0b, y0t = y_positions[g0][s]
            y1b, y1t = y_positions[g1][s]
            path = _ribbon_path(x0, x1, y0b, y0t, y1b, y1t, curvature=curvature, steps=60)
            ax.add_patch(PathPatch(path, facecolor=c, alpha=0.6, edgecolor='none'))

    # Cosmetics
    ax.set_xlim(-gap, 1 + gap)
    ax.set_ylim(0, 1)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(groups)
    ax.set_ylabel("Fraction" if normalize else "Normalized height")
    if title is None:
        title = "Alluvial plot of subtype composition across groups"
    ax.set_title(title)
    ax.grid(False)

    # Labels
    for gi, g in enumerate(groups):
        x = x_positions[gi]
        for s in counts.columns:
            yb, yt = y_positions[g][s]
            frac = yt - yb
            if frac >= min_frac_to_label:
                ax.text(x, yb + frac/2, f"{frac*100:.0f}%", ha='center', va='center', fontsize=8)

    # Legend
    handles = [Rectangle((0,0),1,1, facecolor=subtype_colors[s], edgecolor='none') for s in counts.columns]
    ax.legend(handles, counts.columns.tolist(), title=subtype_col,
              bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0.)

    fig.tight_layout()
    if outfile_prefix:
        fig.savefig(outfile_prefix, bbox_inches='tight', dpi=200)

    return fig, ax, counts, props


In [ ]:
subadata = adata[(adata.obs.Virchow_leiden_joint.isin(['6','15', '16'])) & (adata.obs.refined_celltypes.isin(['Epithelial']))].copy()

In [ ]:
subadata.obs['condition'] = subadata.obs.sample_id.str[-3:]

In [ ]:
subtype_dict = {
    # Secretory lineage
    "Epithelial—Goblet/secretory": "Epithelial—Secretory",
    "Epithelial—crypt/secretory": "Epithelial—Secretory",

    # Progenitor / regenerative / undifferentiated
    "Epithelial—Stem/TA": "Epithelial—Progenitor/Regenerative",
    "Epithelial—TA (pre-absorptive)": "Epithelial—Progenitor/Regenerative",
    "Epithelial—injury/regenerative (LCN2+)": "Epithelial—Progenitor/Regenerative",
    "Epithelial—unspecified": "Epithelial—Progenitor/Regenerative",
    "Epithelial—low-grade/dysplastic": "Epithelial—Progenitor/Regenerative",

    # Mature absorptive lineage (colonocytes)
    "Epithelial—Mature colonocyte (CA+)": "Epithelial—Mature colonocyte",
    "Epithelial—Mature colonocyte I": "Epithelial—Mature colonocyte",
    "Epithelial—Mature colonocyte II": "Epithelial—Mature colonocyte",
    "Epithelial—Mature colonocyte (absorptive)": "Epithelial—Mature colonocyte",
}


In [ ]:
subadata.obs['Cell Subtype'] = subadata.obs.refined_cellsubtypes.replace(subtype_dict)

In [ ]:
palette_specs = {
            'Cell Subtype': tab_filtered,
            'condition': tab_filtered,
            'sample_id': tab_filtered,
        }

palette_dict_2 = build_palettes_from_adata(subadata, palette_specs)

In [ ]:
avg_expr = pd.Series(np.asarray(subadata.X.mean(axis=0)).ravel(), index=subadata.var_names)

hex_subadata = subadata[:,avg_expr[avg_expr>0.5].index].copy()

In [ ]:
sc.tl.rank_genes_groups(hex_subadata, groupby='Virchow_leiden_joint', method='wilcoxon')

In [ ]:
dgex = {}
for gr in hex_subadata.obs.Virchow_leiden_joint.unique():
    dgex[gr] = sc.get.rank_genes_groups_df(hex_subadata, group=gr)

In [ ]:
tmp = subadata.copy()

sc.pp.scale(tmp)

In [ ]:
niche6_genes = dgex['6'][dgex['6']['logfoldchanges']>0].sort_values('logfoldchanges', ascending=False).head(25).names.ravel()
niche15_genes = dgex['15'][dgex['15']['logfoldchanges']>0].sort_values('logfoldchanges', ascending=False).head(25).names.ravel()
niche16_genes = dgex['16'][dgex['16']['logfoldchanges']>0].sort_values('logfoldchanges', ascending=False).head(25).names.ravel()

In [ ]:
genes_to_plot = np.concat([niche6_genes,niche15_genes,niche16_genes])

In [ ]:
sc.pl.dotplot(
    tmp,
    var_names=genes_to_plot,
    groupby='Virchow_leiden_joint',  # must exist in adata.obs
    vmin=-0.25, vmax=0.25,
    cmap="RdBu_r",
    dot_max=0.6,
    dot_min=0.05,
    show=True,
    return_fig=False,
    figsize=(15, 2),
)

In [ ]:
subdf =subadata.obs.copy()
# --- Run demo if user df not present ---
fig, ax, counts_df, props_df = alluvial_multi_groups(
    subdf,
    group_col="Virchow_leiden_joint",
    subtype_col="Cell Subtype",
    palette=palette_dict_2["Cell Subtype"],
    normalize=True,
    title="Epithelial cell composition",
    group_order=['16','15','6'],
    #group_order=[f'{i}' for i in range(18)],
    figsize=(6.5,2),
    outfile_prefix="Virchow_epithelial_alluvial.svg",
)

In [ ]:
subadata = adata[(adata.obs.Virchow_leiden_joint.isin(['6','15', '16'])) & (adata.obs.refined_celltypes.isin(['Lymphoid']))].copy()

In [ ]:
subadata.obs['condition'] = subadata.obs.sample_id.str[-3:]

In [ ]:
celltype_groups = {
    # B lineage
    "B lineage—Plasma cell": "Plasma",
    "B lineage—Plasma cell (activated)": "Plasma",
    "B lineage—activated B / GC-like": "B cell",
    "B lineage—B cell": "B cell",

    # T lineage
    "T cell—naive/Tfh": "T cell",
    "T cell—unspecified": "T cell",
}


In [ ]:
subadata.obs['Cell Subtype'] = subadata.obs.refined_cellsubtypes.replace(celltype_groups)

In [ ]:
palette_specs = {
            'Cell Subtype': tab_filtered,
            'condition': tab_filtered,
            'sample_id': tab_filtered,
        }

palette_dict_2 = build_palettes_from_adata(subadata, palette_specs)

In [ ]:
avg_expr = pd.Series(np.asarray(subadata.X.mean(axis=0)).ravel(), index=subadata.var_names)

hex_subadata = subadata[:,avg_expr[avg_expr>0.5].index].copy()

In [ ]:
sc.tl.rank_genes_groups(hex_subadata, groupby='Virchow_leiden_joint', method='wilcoxon')

In [ ]:
dgex = {}
for gr in hex_subadata.obs.Virchow_leiden_joint.unique():
    dgex[gr] = sc.get.rank_genes_groups_df(hex_subadata, group=gr)

In [ ]:
tmp = subadata.copy()

sc.pp.scale(tmp)

In [ ]:
niche6_genes = dgex['6'][dgex['6']['logfoldchanges']>0].sort_values('logfoldchanges', ascending=False).head(25).names.ravel()
niche15_genes = dgex['15'][dgex['15']['logfoldchanges']>0].sort_values('logfoldchanges', ascending=False).head(25).names.ravel()
niche16_genes = dgex['16'][dgex['16']['logfoldchanges']>0].sort_values('logfoldchanges', ascending=False).head(25).names.ravel()

In [ ]:
genes_to_plot = np.concat([niche6_genes,niche15_genes,niche16_genes])

In [ ]:
sc.pl.dotplot(
    tmp,
    var_names=genes_to_plot,
    groupby='Virchow_leiden_joint',  # must exist in adata.obs
    vmin=-0.25, vmax=0.25,
    cmap="RdBu_r",
    dot_max=0.6,
    dot_min=0.05,
    show=True,
    return_fig=False,
    figsize=(15, 2),
)

In [ ]:
subdf =subadata.obs.copy()
# --- Run demo if user df not present ---
fig, ax, counts_df, props_df = alluvial_multi_groups(
    subdf,
    group_col="Virchow_leiden_joint",
    subtype_col="Cell Subtype",
    palette=palette_dict_2["Cell Subtype"],
    normalize=True,
    title="Lymphoid cell composition",
    group_order=['16','15','6'],
    #group_order=[f'{i}' for i in range(18)],
    figsize=(6.5,2),
    outfile_prefix="Virchow_lymphoid_alluvial.svg",
)

## Res 0.2

In [ ]:
sc.tl.leiden(
    adata,
    resolution=0.2,
    key_added="Virchow_leiden",
    flavor="igraph",
    n_iterations=2
)

In [ ]:
adata.obs["Virchow_leiden"] = (
    adata.obs["Virchow_leiden"].astype("category")
)

print(
    "Number of clusters:",
    adata.obs["Virchow_leiden"].nunique()
)

In [ ]:
adata.obs['Virchow_leiden_joint'] = group_small_clusters(
    adata.obs[['Virchow_leiden']],
    cluster_col='Virchow_leiden',
    min_count= 500,
    new_label= "Other",
    output_col = None
)

In [ ]:
print(
    "Number of clusters:",
    adata.obs["Virchow_leiden_joint"].nunique()
)

In [ ]:
cm = pd.crosstab(
    adata.obs['UNI_leiden_joint'],
    adata.obs['Virchow_leiden_joint']
)

cm_uni = cm.div(cm.sum(axis=0), axis=1)

plt.figure(figsize=(9,8))
sns.heatmap(cm_uni, annot=True, fmt='.2f')
plt.ylabel('UNI cluster')
plt.xlabel('Virchow cluster')
plt.title('Proportion of Virchow cluster assigned to each UNI cluster')

In [ ]:
adata.obs["UNI_leiden_joint"] = (
    adata.obs["UNI_leiden_joint"].astype(str).astype("category")
)

adata.obs["Virchow_leiden_joint"] = (
    adata.obs["Virchow_leiden_joint"].astype(str).astype("category")
)

In [ ]:
import seaborn as sns

all_clusters = sorted(
    set(adata.obs["UNI_leiden_joint"].cat.categories)
    |
    set(adata.obs["Virchow_leiden_joint"].cat.categories)
)

palette = sns.color_palette(
    "tab20",
    n_colors=len(all_clusters)
)

cluster_colors = {
    cl: palette[i]
    for i, cl in enumerate(all_clusters)
}

In [ ]:
fig, axes = plt.subplots(
    1,
    len(samples),
    figsize=(20,4)
)

for ax, sample in zip(axes, samples):

    ad = adata[adata.obs.sample_id == sample]

    sc.pl.embedding(
        ad,
        basis="spatial",
        color="Virchow_leiden_joint",
        palette=cluster_colors,
        ax=ax,
        show=False,
        frameon=False,
        size=8,
        legend_loc=None
    )

    ax.set_title(sample)

plt.tight_layout()

## Res 0.25

In [ ]:
sc.tl.leiden(
    adata,
    resolution=0.25,
    key_added="Virchow_leiden",
    flavor="igraph",
    n_iterations=2
)

In [ ]:
adata.obs["Virchow_leiden"] = (
    adata.obs["Virchow_leiden"].astype("category")
)

print(
    "Number of clusters:",
    adata.obs["Virchow_leiden"].nunique()
)

In [ ]:
adata.obs['Virchow_leiden_joint'] = group_small_clusters(
    adata.obs[['Virchow_leiden']],
    cluster_col='Virchow_leiden',
    min_count= 500,
    new_label= "Other",
    output_col = None
)

In [ ]:
print(
    "Number of clusters:",
    adata.obs["Virchow_leiden_joint"].nunique()
)

In [ ]:
cm = pd.crosstab(
    adata.obs['UNI_leiden_joint'],
    adata.obs['Virchow_leiden_joint']
)

cm_uni = cm.div(cm.sum(axis=0), axis=1)

plt.figure(figsize=(9,8))
sns.heatmap(cm_uni, annot=True, fmt='.2f')
plt.ylabel('UNI cluster')
plt.xlabel('Virchow cluster')
plt.title('Proportion of Virchow cluster assigned to each UNI cluster')

In [ ]:
adata.obs["UNI_leiden_joint"] = (
    adata.obs["UNI_leiden_joint"].astype(str).astype("category")
)

adata.obs["Virchow_leiden_joint"] = (
    adata.obs["Virchow_leiden_joint"].astype(str).astype("category")
)

In [ ]:
import seaborn as sns

all_clusters = sorted(
    set(adata.obs["UNI_leiden_joint"].cat.categories)
    |
    set(adata.obs["Virchow_leiden_joint"].cat.categories)
)

palette = sns.color_palette(
    "tab20",
    n_colors=len(all_clusters)
)

cluster_colors = {
    cl: palette[i]
    for i, cl in enumerate(all_clusters)
}

In [ ]:
fig, axes = plt.subplots(
    1,
    len(samples),
    figsize=(20,4)
)

for ax, sample in zip(axes, samples):

    ad = adata[adata.obs.sample_id == sample]

    sc.pl.embedding(
        ad,
        basis="spatial",
        color="Virchow_leiden_joint",
        palette=cluster_colors,
        ax=ax,
        show=False,
        frameon=False,
        size=8,
        legend_loc=None
    )

    ax.set_title(sample)

plt.tight_layout()

In [ ]:
sc.tl.leiden(
    adata,
    resolution=0.3,
    key_added="Virchow_leiden",
    flavor="igraph",
    n_iterations=2
)

In [ ]:
adata.obs["Virchow_leiden"] = (
    adata.obs["Virchow_leiden"].astype("category")
)

print(
    "Number of clusters:",
    adata.obs["Virchow_leiden"].nunique()
)

In [ ]:
adata.obs['Virchow_leiden_joint'] = group_small_clusters(
    adata.obs[['Virchow_leiden']],
    cluster_col='Virchow_leiden',
    min_count= 500,
    new_label= "Other",
    output_col = None
)

In [ ]:
print(
    "Number of clusters:",
    adata.obs["Virchow_leiden_joint"].nunique()
)

In [ ]:
cm = pd.crosstab(
    adata.obs['UNI_leiden_joint'],
    adata.obs['Virchow_leiden_joint']
)

cm_uni = cm.div(cm.sum(axis=0), axis=1)

plt.figure(figsize=(9,8))
sns.heatmap(cm_uni, annot=True, fmt='.2f')
plt.ylabel('UNI cluster')
plt.xlabel('Virchow cluster')
plt.title('Proportion of Virchow cluster assigned to each UNI cluster')

In [ ]:
adata.obs["UNI_leiden_joint"] = (
    adata.obs["UNI_leiden_joint"].astype(str).astype("category")
)

adata.obs["Virchow_leiden_joint"] = (
    adata.obs["Virchow_leiden_joint"].astype(str).astype("category")
)

In [ ]:
import seaborn as sns

all_clusters = sorted(
    set(adata.obs["UNI_leiden_joint"].cat.categories)
    |
    set(adata.obs["Virchow_leiden_joint"].cat.categories)
)

palette = sns.color_palette(
    "tab20",
    n_colors=len(all_clusters)
)

cluster_colors = {
    cl: palette[i]
    for i, cl in enumerate(all_clusters)
}

In [ ]:
fig, axes = plt.subplots(
    1,
    len(samples),
    figsize=(20,4)
)

for ax, sample in zip(axes, samples):

    ad = adata[adata.obs.sample_id == sample]

    sc.pl.embedding(
        ad,
        basis="spatial",
        color="Virchow_leiden_joint",
        palette=cluster_colors,
        ax=ax,
        show=False,
        frameon=False,
        size=8,
        legend_loc=None
    )

    ax.set_title(sample)

plt.tight_layout()

In [ ]:
sc.tl.leiden(
    adata,
    resolution=0.35,
    key_added="Virchow_leiden",
    flavor="igraph",
    n_iterations=2
)

In [ ]:
adata.obs["Virchow_leiden"] = (
    adata.obs["Virchow_leiden"].astype("category")
)

print(
    "Number of clusters:",
    adata.obs["Virchow_leiden"].nunique()
)

In [ ]:
adata.obs['Virchow_leiden_joint'] = group_small_clusters(
    adata.obs[['Virchow_leiden']],
    cluster_col='Virchow_leiden',
    min_count= 500,
    new_label= "Other",
    output_col = None
)

In [ ]:
print(
    "Number of clusters:",
    adata.obs["Virchow_leiden_joint"].nunique()
)

In [ ]:
cm = pd.crosstab(
    adata.obs['UNI_leiden_joint'],
    adata.obs['Virchow_leiden_joint']
)

cm_uni = cm.div(cm.sum(axis=0), axis=1)

plt.figure(figsize=(9,8))
sns.heatmap(cm_uni, annot=True, fmt='.2f')
plt.ylabel('UNI cluster')
plt.xlabel('Virchow cluster')
plt.title('Proportion of Virchow cluster assigned to each UNI cluster')

In [ ]:
adata.obs["UNI_leiden_joint"] = (
    adata.obs["UNI_leiden_joint"].astype(str).astype("category")
)

adata.obs["Virchow_leiden_joint"] = (
    adata.obs["Virchow_leiden_joint"].astype(str).astype("category")
)

In [ ]:
import seaborn as sns

all_clusters = sorted(
    set(adata.obs["UNI_leiden_joint"].cat.categories)
    |
    set(adata.obs["Virchow_leiden_joint"].cat.categories)
)

palette = sns.color_palette(
    "tab20",
    n_colors=len(all_clusters)
)

cluster_colors = {
    cl: palette[i]
    for i, cl in enumerate(all_clusters)
}

In [ ]:
fig, axes = plt.subplots(
    1,
    len(samples),
    figsize=(20,4)
)

for ax, sample in zip(axes, samples):

    ad = adata[adata.obs.sample_id == sample]

    sc.pl.embedding(
        ad,
        basis="spatial",
        color="Virchow_leiden_joint",
        palette=cluster_colors,
        ax=ax,
        show=False,
        frameon=False,
        size=8,
        legend_loc=None
    )

    ax.set_title(sample)

plt.tight_layout()